# §6.1 recovery sweep figures

Loads `recovery_sweep.py` outputs and Phase A cells; produces the four §6.1 figures via `src_tb.support_recovery.visualization`.

- Phase A cache: `experiments/causal_prior/synthetic/cache_p30_headline/`
- Recovery sweep output: `experiments/causal_prior/synthetic/recovery_p30_headline/`

In [ ]:
from pathlib import Path
import sys

# project root on sys.path so src_tb imports work from a notebook in a subdir
ROOT = Path.cwd().resolve()
while not (ROOT / 'src_tb').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src_tb.support_recovery.loading import (
    load_recovery_csvs, load_phase_a_npz, selectivity_per_cell, aggregate_seeds,
)
from src_tb.support_recovery import visualization as viz

In [ ]:
CACHE = ROOT / 'experiments/causal_prior/synthetic/cache_p30_headline'
OUT   = ROOT / 'experiments/causal_prior/synthetic/recovery_p30_headline'

recovery = load_recovery_csvs(OUT)
phase_a  = load_phase_a_npz(CACHE)
sel      = selectivity_per_cell(phase_a)
merged   = recovery.merge(sel, on=['seed', 'p', 'n', 'k_star', 'p_edge', 'q_source'], how='left')

print(f'recovery rows: {len(recovery)}, cells: {recovery.groupby(["seed","p","n","k_star","p_edge"]).ngroups}')
print(f'phase A cells: {len(phase_a)}')
print(f'sources present: {sorted(recovery.q_source.unique())}')

## anchor cell sanity check: precision table at $p_{\mathrm{edge}} = 0.2$

Quick numeric look before plotting. Expect: oracle and GES climb; uniform invariant; adversarial collapse; L1 modestly degrades.

In [ ]:
anchor = recovery[(recovery.p == 30) & (recovery.n == 300) & (recovery.k_star == 5) & (recovery.p_edge == 0.2)]
soft = anchor[~anchor.q_source.str.contains('_hard_')]
table = (soft.groupby(['q_source', 'mu_relative'])
             .S_precision.mean().unstack().round(2))
table

## Fig 1A: anchor-cell $\mu$-sensitivity

In [ ]:
anchor_rows = recovery[(recovery.p == 30) & (recovery.n == 300) & (recovery.k_star == 5)]

fig, ax = plt.subplots(figsize=(7, 5))
viz.plot_recovery_vs_mu(anchor_rows, ax, p_edge=0.2, metric='S_precision')
plt.tight_layout()
plt.show()

## Fig 1B: $p_{\mathrm{edge}}$ facet at the headline shape

In [ ]:
p_edges = sorted(anchor_rows.p_edge.unique())
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
viz.plot_recovery_vs_mu_facet(anchor_rows, axes.flatten(), metric='S_precision', p_edges=p_edges)
plt.tight_layout()
plt.show()

## Fig 2: recovery vs selectivity

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
viz.plot_recovery_vs_selectivity(merged[(merged.p == 30) & (merged.n == 300) & (merged.k_star == 5)],
                                  ax, mu_relative_ref=1.0)
plt.tight_layout()
plt.show()

## Fig 3: soft vs hard prior comparison (§6.4 baseline)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, src in zip(axes, ['pc', 'ges', 'bootstrap_l1']):
    try:
        viz.plot_soft_vs_hard(anchor_rows, ax, q_source=src, metric='S_precision')
    except ValueError as e:
        ax.set_title(f'{src}: no data')
        print(f'{src}: {e}')
plt.tight_layout()
plt.show()